In [ ]:
# ============================================================================
# CELL 0 — Install Dependencies
# ============================================================================
# Run this cell once. Skip if already installed.
# ============================================================================

import subprocess, sys

packages = [
    'torch', 'torchvision',
    'scikit-learn', 'matplotlib',
    'requests', 'Pillow'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('✓ All dependencies installed')

In [ ]:
# ============================================================================
# CELL 1 — Imports
# ============================================================================

import os
import json
import time
import hashlib
import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

print(f'✓ PyTorch {torch.__version__}')
print(f'  CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ============================================================================
# CELL 2 — Configuration
# ============================================================================
# CHANGE THESE FOR EACH HOSPITAL
# ============================================================================

# ── Which hospital is this? ───────────────────────────────────────────────────
# Options: hospital_1  hospital_2  hospital_3  hospital_4  hospital_5
HOSPITAL_ID = 'hospital_1'

# ── Path to THIS hospital's local dataset ────────────────────────────────────
# After running scripts/split_dataset.py, each hospital has its own folder.
# Example: './data/hospitals/hospital_1'
# The folder must contain: train/NORMAL/, train/PNEUMONIA/, val/, test/
LOCAL_DATASET_PATH = './data/hospitals/hospital_1'

# ── Central coordination server ──────────────────────────────────────────────
SERVER_URL = 'http://localhost:5000'

# ── Which FL round is this? ──────────────────────────────────────────────────
# Round 1: First training run (starts from pretrained ResNet-18)
# Round 2+: Downloads aggregated global model first, then fine-tunes
CURRENT_ROUND = 1

# ── Training hyperparameters ──────────────────────────────────────────────────
LOCAL_EPOCHS   = 5      # Epochs to train locally per round
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-4

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load API key for this hospital ───────────────────────────────────────────
# Reads from config/hospitals.json (relative to project root)
# If you're running the notebook from a different directory, adjust the path.
config_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'config', 'hospitals.json')
if not os.path.exists(config_path):
    # Try relative path (if running from project root)
    config_path = './config/hospitals.json'

try:
    with open(config_path, 'r') as f:
        hospital_tokens = json.load(f)
    API_KEY = hospital_tokens.get(HOSPITAL_ID)
    if not API_KEY:
        raise KeyError(f'No token found for {HOSPITAL_ID}')
    print(f'✓ API key loaded for {HOSPITAL_ID}')
except FileNotFoundError:
    print(f'⚠️  config/hospitals.json not found at {config_path}')
    print('   Set API_KEY manually below.')
    API_KEY = 'hospital1_token_abc123def456'  # fallback for hospital_1

print(f'''
Configuration:
  Hospital ID  : {HOSPITAL_ID}
  Dataset Path : {LOCAL_DATASET_PATH}
  Server       : {SERVER_URL}
  FL Round     : {CURRENT_ROUND}
  Device       : {device}
  Epochs       : {LOCAL_EPOCHS}

⚠️  PRIVACY: Patient data will NOT leave this machine.
   Only model weights and summary metrics are sent to the server.
''')

In [ ]:
# ============================================================================
# CELL 3 — Download Global Model from Server (Round >= 2 only)
# ============================================================================
# Round 1: No global model exists yet. Skip this cell.
# Round N>=2: Downloads global_model_round(N-1).pt from the server.
#             This is the aggregated model from the previous round.
# ============================================================================

global_model_path   = None
downloaded_round    = None
download_successful = False

if CURRENT_ROUND == 1:
    print('Round 1: Starting from pretrained ResNet-18. No global model to download.')

else:
    print(f'Round {CURRENT_ROUND}: Downloading aggregated global model from server...')
    try:
        response = requests.get(
            f'{SERVER_URL}/api/global-model/latest',
            timeout=60,
            stream=True
        )

        if response.status_code == 200:
            round_from_header = response.headers.get('X-GLOBAL-ROUND', 'unknown')
            model_hash_header = response.headers.get('X-MODEL-HASH', '')

            save_path = f'global_model_round{round_from_header}.pt'

            # Save the file
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            # Verify SHA-256 integrity
            sha256 = hashlib.sha256()
            with open(save_path, 'rb') as f:
                for chunk in iter(lambda: f.read(4096), b''):
                    sha256.update(chunk)
            local_hash = sha256.hexdigest()

            if model_hash_header and local_hash != model_hash_header:
                print(f'⚠️  Hash mismatch! File may be corrupted.')
                print(f'   Server hash: {model_hash_header}')
                print(f'   Local hash:  {local_hash}')
            else:
                print(f'✓ Global model downloaded and verified')
                print(f'  File: {save_path}')
                print(f'  Round: {round_from_header}')
                print(f'  SHA-256: {local_hash[:32]}...')

            global_model_path   = save_path
            downloaded_round    = int(round_from_header)
            download_successful = True

        elif response.status_code == 404:
            print('⚠️  No global model available yet on server.')
            print('   Will start from fresh pretrained model.')

        else:
            print(f'⚠️  Server returned HTTP {response.status_code}.')
            print('   Will start from fresh pretrained model.')

    except requests.exceptions.ConnectionError:
        print(f'⚠️  Cannot connect to {SERVER_URL}.')
        print('   Is the Flask server running? (python app.py)')
        print('   Will start from fresh pretrained model.')

    except Exception as e:
        print(f'⚠️  Download failed: {str(e)}')
        print('   Will start from fresh pretrained model.')

In [ ]:
# ============================================================================
# CELL 4 — Load Local Dataset
# ============================================================================
# Loads data from LOCAL_DATASET_PATH.
# Required folder structure:
#   LOCAL_DATASET_PATH/
#     train/
#       NORMAL/     ← chest X-rays of healthy patients
#       PNEUMONIA/  ← chest X-rays of pneumonia patients
#     val/
#       NORMAL/
#       PNEUMONIA/
#     test/
#       NORMAL/
#       PNEUMONIA/
#
# ⚠️  PRIVACY: This data stays local. It is NEVER sent to the server.
# ============================================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dir = os.path.join(LOCAL_DATASET_PATH, 'train')
val_dir   = os.path.join(LOCAL_DATASET_PATH, 'val')
test_dir  = os.path.join(LOCAL_DATASET_PATH, 'test')

for d, name in [(train_dir, 'train'), (val_dir, 'val'), (test_dir, 'test')]:
    if not os.path.exists(d):
        raise FileNotFoundError(
            f'Missing directory: {d}\n'
            f'Run: python scripts/split_dataset.py --source /path/to/chest_xray --dest ./data/hospitals'
        )

train_dataset = ImageFolder(train_dir, transform=transform)
val_dataset   = ImageFolder(val_dir,   transform=transform)
test_dataset  = ImageFolder(test_dir,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_TRAIN_SAMPLES = len(train_dataset)
class_names = train_dataset.classes  # ['NORMAL', 'PNEUMONIA']

print(f'Dataset loaded for {HOSPITAL_ID}:')
print(f'  Train:  {len(train_dataset):,} images')
print(f'  Val:    {len(val_dataset):,} images')
print(f'  Test:   {len(test_dataset):,} images')
print(f'  Classes: {class_names}')
print(f'\n⚠️  PRIVACY: These {NUM_TRAIN_SAMPLES:,} training images stay local to {HOSPITAL_ID}.')
print(f'   Only model weights and aggregate metrics will be sent to the server.')

In [ ]:
# ============================================================================
# CELL 5 — Define Model Architecture
# ============================================================================
# All hospitals use the same architecture (ResNet-18, 2 output classes).
# This is required for FedAvg — models must have identical parameter shapes.
# ============================================================================

class PneumoniaModel(nn.Module):
    """
    ResNet-18 fine-tuned for binary pneumonia classification.
    Input:  224×224 RGB chest X-ray
    Output: 2 logits [NORMAL, PNEUMONIA]
    """
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        # Replace final FC layer: 512 → 2 classes
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

    def forward(self, x):
        return self.model(x)

print('✓ PneumoniaModel defined (ResNet-18, 2 classes)')
print(f'  Architecture: ResNet-18 + FC(512→2)')

In [ ]:
# ============================================================================
# CELL 6 — Define Evaluation Function
# ============================================================================

def evaluate_model(model, dataloader, device):
    """
    Evaluate model on a dataloader.
    Returns: accuracy, loss, precision, recall, f1, auc_roc
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    y_true, y_pred, y_proba = [], [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            proba       = torch.softmax(outputs, dim=1)
            preds       = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_proba.extend(proba[:, 1].cpu().numpy())  # probability of PNEUMONIA class

    n        = len(y_true)
    avg_loss = total_loss / n if n > 0 else 0

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc_roc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc_roc = 0.0

    return {
        'accuracy':  round(accuracy,  4),
        'loss':      round(avg_loss,  4),
        'precision': round(precision, 4),
        'recall':    round(recall,    4),
        'f1_score':  round(f1,        4),
        'auc_roc':   round(auc_roc,   4),
        'num_samples': n
    }

print('✓ evaluate_model() defined')

In [ ]:
# ============================================================================
# CELL 7 — Initialize Model for This Round
# ============================================================================
# Round 1: Fresh pretrained ResNet-18
# Round N>=2: Load the server-aggregated global model from Cell 3
#             Falls back to fresh model if download failed
# ============================================================================

if CURRENT_ROUND == 1:
    model = PneumoniaModel().to(device)
    print(f'✓ Round 1: Initialized fresh ResNet-18 (ImageNet pretrained)')
    print(f'  This hospital will fine-tune from scratch on its local data.')

else:
    model = PneumoniaModel().to(device)

    if download_successful and global_model_path and os.path.exists(global_model_path):
        try:
            state_dict = torch.load(global_model_path, map_location=device)
            model.load_state_dict(state_dict)
            print(f'✓ Loaded server-aggregated global model from round {downloaded_round}')
            print(f'  File: {global_model_path}')
            print(f'  This hospital continues fine-tuning from the global model.')
        except Exception as e:
            print(f'⚠️  Failed to load global model: {str(e)}')
            print(f'   Falling back to fresh pretrained model.')
    else:
        print(f'⚠️  No downloaded global model available.')
        print(f'   Starting from fresh pretrained ResNet-18.')

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n  Trainable parameters: {total_params:,}')

In [ ]:
# ============================================================================
# CELL 8 — Local Training
# ============================================================================
# Trains the model on THIS hospital's local data only.
# Patient images never leave this machine.
# ============================================================================

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

print(f'Training {HOSPITAL_ID} — Round {CURRENT_ROUND}')
print(f'  Epochs: {LOCAL_EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}')
print(f'  Training on: {device}')
print('─' * 60)

train_losses    = []
train_accuracies = []

for epoch in range(1, LOCAL_EPOCHS + 1):
    model.train()
    epoch_loss    = 0.0
    correct       = 0
    total         = 0
    epoch_start   = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * images.size(0)
        preds       = torch.argmax(outputs, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        # Progress indicator every 10 batches
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(train_loader):
            print(f'  Epoch {epoch}/{LOCAL_EPOCHS} [{batch_idx+1}/{len(train_loader)}] '
                  f'loss={loss.item():.4f}', end='\r')

    avg_loss = epoch_loss / total
    accuracy = correct / total
    elapsed  = time.time() - epoch_start

    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)

    # Validation
    val_metrics = evaluate_model(model, val_loader, device)

    print(f'  Epoch {epoch}/{LOCAL_EPOCHS} '
          f'| train_loss={avg_loss:.4f} train_acc={accuracy*100:.2f}% '
          f'| val_loss={val_metrics["loss"]:.4f} val_acc={val_metrics["accuracy"]*100:.2f}% '
          f'| {elapsed:.1f}s')

print('─' * 60)
print(f'✓ Training complete — {LOCAL_EPOCHS} epochs on {NUM_TRAIN_SAMPLES:,} samples')

In [ ]:
# ============================================================================
# CELL 9 — Evaluate on Test Set
# ============================================================================

print(f'Evaluating {HOSPITAL_ID} on local test set...')
test_metrics = evaluate_model(model, test_loader, device)
test_metrics['num_samples'] = NUM_TRAIN_SAMPLES  # report training sample count for FedAvg weighting

print(f'''
=== Test Results — {HOSPITAL_ID} Round {CURRENT_ROUND} ===
  Accuracy  : {test_metrics["accuracy"]*100:.2f}%
  Precision : {test_metrics["precision"]*100:.2f}%
  Recall    : {test_metrics["recall"]*100:.2f}%
  F1 Score  : {test_metrics["f1_score"]:.4f}
  AUC-ROC   : {test_metrics["auc_roc"]:.4f}
  Loss      : {test_metrics["loss"]:.4f}
  Train Samples: {NUM_TRAIN_SAMPLES:,}  (used for FedAvg weighting)
''')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, LOCAL_EPOCHS+1), train_losses, 'b-o', label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'{HOSPITAL_ID} — Training Loss (Round {CURRENT_ROUND})')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(range(1, LOCAL_EPOCHS+1), [a*100 for a in train_accuracies], 'g-o', label='Train Acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title(f'{HOSPITAL_ID} — Training Accuracy (Round {CURRENT_ROUND})')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{HOSPITAL_ID}_round{CURRENT_ROUND}_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'✓ Training curves saved to {HOSPITAL_ID}_round{CURRENT_ROUND}_curves.png')

In [ ]:
# ============================================================================
# CELL 10 — Save Local Weights
# ============================================================================
# Saves ONLY the model weights (state_dict).
# This file is what gets submitted to the server.
# It contains parameter tensors ONLY — no patient data, no images.
# ============================================================================

weights_filename = f'local_weights_{HOSPITAL_ID}_round{CURRENT_ROUND}.pt'
torch.save(model.state_dict(), weights_filename)

# Compute SHA-256 for integrity verification
sha256 = hashlib.sha256()
with open(weights_filename, 'rb') as f:
    for chunk in iter(lambda: f.read(4096), b''):
        sha256.update(chunk)
weights_hash = sha256.hexdigest()

file_size_mb = os.path.getsize(weights_filename) / (1024 * 1024)

print(f'✓ Local weights saved')
print(f'  File      : {weights_filename}')
print(f'  Size      : {file_size_mb:.2f} MB')
print(f'  SHA-256   : {weights_hash[:32]}...')
print(f'\n⚠️  Privacy check: This file contains only model parameters.')
print(f'   It does NOT contain any patient images or medical records.')

In [ ]:
# ============================================================================
# CELL 11 — Build Submission Payload
# ============================================================================

submission_metrics = {
    'accuracy':    test_metrics['accuracy'],
    'loss':        test_metrics['loss'],
    'num_samples': NUM_TRAIN_SAMPLES,       # ← REQUIRED for correct FedAvg weighting
    'precision':   test_metrics['precision'],
    'recall':      test_metrics['recall'],
    'f1_score':    test_metrics['f1_score'],
    'auc_roc':     test_metrics['auc_roc'],
}

print('Submission payload:')
print(f'  hospital_id  : {HOSPITAL_ID}')
print(f'  round        : {CURRENT_ROUND}')
print(f'  weights file : {weights_filename} ({file_size_mb:.2f} MB)')
print(f'  metrics      : {json.dumps(submission_metrics, indent=4)}')

In [ ]:
# ============================================================================
# CELL 12 — Submit to Server
# ============================================================================
# Sends weights.pt + metrics JSON to the central server.
# Server validates, stores, and triggers FedAvg when all 5 hospitals submit.
# ============================================================================

print(f'Submitting to {SERVER_URL}/api/submit_update ...')

try:
    with open(weights_filename, 'rb') as weights_file:
        response = requests.post(
            f'{SERVER_URL}/api/submit_update',
            files={'weights': (weights_filename, weights_file, 'application/octet-stream')},
            data={
                'hospital_id': HOSPITAL_ID,
                'round':       str(CURRENT_ROUND),
                'metrics':     json.dumps(submission_metrics)
            },
            headers={'X-API-KEY': API_KEY},
            timeout=120
        )

    if response.status_code == 200:
        result = response.json()
        agg_status = result.get('aggregation_status', 'unknown')
        received   = result.get('submissions_received', '?')
        expected   = result.get('submissions_expected', '?')
        weights_hash_server = result.get('weights_hash', 'not returned')[:32]

        print(f'''
✓ Submission accepted by server
  Status         : {agg_status}
  Submissions    : {received} / {expected} hospitals
  Weights hash   : {weights_hash_server}...
''')

        if agg_status == 'round_complete':
            print('🎉 ALL HOSPITALS SUBMITTED — FedAvg aggregation triggered!')
            print('   Check the Admin Dashboard and Blockchain Ledger in the browser.')
        else:
            print(f'   Waiting for {int(expected) - int(received)} more hospital(s) to submit.')

    elif response.status_code == 409:
        print(f'⚠️  Already submitted for round {CURRENT_ROUND}. Duplicate rejected.')

    elif response.status_code == 401:
        print(f'✗ Authentication failed. Check API_KEY for {HOSPITAL_ID}.')
        print(f'  Response: {response.json()}')

    else:
        print(f'✗ Submission failed — HTTP {response.status_code}')
        print(f'  Response: {response.text[:200]}')

except requests.exceptions.ConnectionError:
    print(f'✗ Cannot connect to {SERVER_URL}.')
    print('  Make sure the Flask server is running: python app.py')

except Exception as e:
    print(f'✗ Submission error: {str(e)}')

In [ ]:
# ============================================================================
# CELL 13 — Summary
# ============================================================================

print('=' * 60)
print(f'  FL ROUND {CURRENT_ROUND} COMPLETE — {HOSPITAL_ID}')
print('=' * 60)
print(f'''
  What happened:
    ✓ Loaded local dataset ({NUM_TRAIN_SAMPLES:,} training samples)
    ✓ Trained ResNet-18 for {LOCAL_EPOCHS} epochs locally
    ✓ Evaluated: accuracy={test_metrics["accuracy"]*100:.2f}%, loss={test_metrics["loss"]:.4f}
    ✓ Saved weights to {weights_filename}
    ✓ Submitted weights + metrics to server

  What NEVER happened:
    ✗ Patient images were NOT uploaded
    ✗ Raw data was NOT shared
    ✗ Data left the hospital machine

  Next steps:
    1. Open other hospital notebooks and run them for round {CURRENT_ROUND}
    2. After all 5 hospitals submit, FedAvg runs automatically on the server
    3. Set CURRENT_ROUND = {CURRENT_ROUND + 1} in Cell 2 for the next round
    4. Re-run notebook — Cell 3 will download the aggregated global model
''')
print('=' * 60)